# Whisper paraphasia detection — Scripts-Fridriksson LOSO on Kaggle (2× T4)

Data: private Kaggle dataset `evandiewald/aphasia-scripts-fridriksson` (fetched with kagglehub).
Needs the `WANDB_API_KEY` secret (metrics logging only), internet on, and Persistence → Files on.

Folds train in a background process, one per GPU, with checkpoints in `/root` scratch. Each finished fold
is copied to `/kaggle/working/checkpoints/<RUN_NAME>` (persisted, 20 GB cap); re-running the launch cell
skips folds already there. Keep the watch cell running while training: Kaggle shuts down interactive
sessions it thinks are idle, and a background process doesn't count as activity.

In [ ]:
import os, subprocess, json, time, re, glob

REPO_DIR = '/kaggle/working/aphasia-modeling'
DATA_PATH = 'datasets/Scripts/scripts_fridriksson.json'

if os.path.exists(REPO_DIR):
    !cd {REPO_DIR} && git pull -q
else:
    !git clone -q https://github.com/evandiewald/aphasia-modeling.git {REPO_DIR}
!cd {REPO_DIR} && git log --oneline -1 && pip install -e . -q 2>&1 | grep -v 'requires\|incompatible\|resolver'

from kaggle_secrets import UserSecretsClient
os.environ['WANDB_API_KEY'] = UserSecretsClient().get_secret('WANDB_API_KEY')

# Link the dataset into the repo at the paths the JSON references
import kagglehub
src = kagglehub.dataset_download('evandiewald/aphasia-scripts-fridriksson')
audio_dir = f'{REPO_DIR}/datasets/Scripts/Fridriksson/audio'
os.makedirs(os.path.dirname(audio_dir), exist_ok=True)
if not os.path.islink(audio_dir):
    os.symlink(f'{src}/audio', audio_dir)
!cp {src}/scripts_fridriksson.json {REPO_DIR}/{DATA_PATH}
records = json.load(open(f'{REPO_DIR}/{DATA_PATH}'))
missing = {r['audio_path'] for r in records if not os.path.exists(f"{REPO_DIR}/{r['audio_path']}")}
print(f"{len(records)} utterances, {len(missing)} missing audio files")
assert not missing, sorted(missing)[:5]
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
RUN_NAME = 'scripts-small-pn'
TAG_CLASSES = 'pn'        # 'pns' to train [s] too
SPEAKERS = 'P4,P10'       # representative moderate folds; '' for all 12
SCRATCH = f'/root/checkpoints/{RUN_NAME}'             # in-progress checkpoints (~3 GB/fold)
KEEP = f'/kaggle/working/checkpoints/{RUN_NAME}'      # finished folds (~1 GB each), persisted
RESULTS = f'/kaggle/working/results/{RUN_NAME}'
for d in (SCRATCH, KEEP, RESULTS):
    os.makedirs(d, exist_ok=True)

In [ ]:
# Launch in the background, one fold per GPU (T4: fp16, batch 4 x accum 4 = effective 16)
if subprocess.run(['pgrep', '-f', f'run_name {RUN_NAME}( |$)'], capture_output=True).returncode == 0:
    print('Runner already running')
else:
    speakers = f'--speakers {SPEAKERS} ' if SPEAKERS else ''
    cmd = (f'cd {REPO_DIR} && nohup python scripts/run_loso_parallel.py --gpus 0,1 {speakers}'
           f'--data_path {DATA_PATH} --output_dir {SCRATCH} --keep_dir {KEEP} --run_name {RUN_NAME} -- '
           f'--tag_classes {TAG_CLASSES} --fp16 --batch_size 4 --grad_accum 4 --num_workers 2 --wandb '
           f'>> {SCRATCH}/runner.log 2>&1 &')
    subprocess.run(cmd, shell=True, start_new_session=True)
    print(f'Started; runner log: {SCRATCH}/runner.log')

In [ ]:
# Watch until the runner exits (keeps the session active). Safe to interrupt and re-run.
def status():
    print(time.strftime('%H:%M:%S'), open(f'{SCRATCH}/runner.log').read().strip().splitlines()[-1])
    for log in sorted(glob.glob(f'{SCRATCH}/logs/*.log')):
        txt = open(log).read()
        evals = re.findall(r"\{'eval_loss'.*?'epoch': '[\d.]+'\}", txt)
        last = evals[-1] if evals else ''
        g = lambda k: (re.search(rf"'eval_{k}': '?([\d.]+)", last) or [None, '?'])[1]
        flag = ' DONE' if 'Fold complete' in txt else (' TRACEBACK' if 'Traceback' in txt else '')
        print(f"  {os.path.basename(log):15s} epoch {len(evals):2d} loss {g('loss')} "
              f"pred/ref p {g('pred_p')}/{g('ref_p')} n {g('pred_n')}/{g('ref_n')} s {g('pred_s')}/{g('ref_s')} "
              f"tag acc {g('tag_class_acc')}{flag}")

while subprocess.run(['pgrep', '-f', f'run_name {RUN_NAME}( |$)'], capture_output=True).returncode == 0:
    status()
    time.sleep(300)
status()

In [ ]:
# Evaluate the kept folds on their held-out speakers (pooled + per fold)
speakers = f'--speakers {SPEAKERS}' if SPEAKERS else ''
!cd {REPO_DIR} && python scripts/evaluate_model.py --model_dir {KEEP} --data_path {DATA_PATH} --loso {speakers} \
    --output_dir {RESULTS} --batch_size 16 2>&1 | grep -v 'it/s'

In [ ]:
# Compare against the pns run on the same folds, scoring only [p]/[n] so neither run is charged for [s].
# Baseline predictions come from /kaggle/working if persisted, else its W&B checkpoints are re-evaluated.
BASE = 'scripts-small-taginit'
BASE_RESULTS = f'/kaggle/working/results/{BASE}'
for spk in (SPEAKERS or '').split(','):
    if spk and not os.path.exists(f'{BASE_RESULTS}/fold_{spk}/predictions.json'):
        !cd {REPO_DIR} && python scripts/run_loso_parallel.py --download --data_path {DATA_PATH} \
            --output_dir /root/checkpoints/{BASE} --run_name {BASE} --speakers {spk}
        !cd {REPO_DIR} && python scripts/evaluate_model.py --model_dir /root/checkpoints/{BASE}/fold_{spk} \
            --data_path {DATA_PATH} --test_speaker {spk} --output_dir {BASE_RESULTS}/fold_{spk} --batch_size 16 2>&1 | tail -1
speakers = f'--speakers {SPEAKERS}' if SPEAKERS else ''
!cd {REPO_DIR} && python scripts/score_predictions.py {BASE_RESULTS} {RESULTS} {speakers} --classes pn --per_speaker